In [2]:
from pathlib import Path
import random
import shutil
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms


def is_oxford_pet_ready(data_dir: Path) -> bool:
    pet_dir = data_dir / "oxford-iiit-pet"
    images_dir = pet_dir / "images"
    annotations_dir = pet_dir / "annotations"
    has_images = images_dir.is_dir() and any(images_dir.iterdir())
    has_annotations = annotations_dir.is_dir() and any(annotations_dir.iterdir())
    return has_images and has_annotations


def load_oxford_pet_datasets(data_dir: Path, train_transform, eval_transform):
    already_present = is_oxford_pet_ready(data_dir)
    print(f"Oxford-IIIT Pet already present at {data_dir}: {already_present}")

    try:
        train_ds = datasets.OxfordIIITPet(
            root=data_dir,
            split="trainval",
            target_types="category",
            download=not already_present,
            transform=train_transform,
        )
        test_ds = datasets.OxfordIIITPet(
            root=data_dir,
            split="test",
            target_types="category",
            download=not already_present,
            transform=eval_transform,
        )
        return train_ds, test_ds
    except Exception as exc:
        print(f"Dataset load/extract failed ({exc}). Cleaning partial files and retrying download...")
        pet_dir = data_dir / "oxford-iiit-pet"
        if pet_dir.exists():
            shutil.rmtree(pet_dir, ignore_errors=True)
        train_ds = datasets.OxfordIIITPet(
            root=data_dir,
            split="trainval",
            target_types="category",
            download=True,
            transform=train_transform,
        )
        test_ds = datasets.OxfordIIITPet(
            root=data_dir,
            split="test",
            target_types="category",
            download=True,
            transform=eval_transform,
        )
        return train_ds, test_ds


def make_subset(dataset, max_samples: int, seed: int = 42) -> Subset:
    rng = random.Random(seed)
    indices = list(range(len(dataset)))
    rng.shuffle(indices)
    return Subset(dataset, indices[: min(max_samples, len(dataset))])


def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> tuple[float, float]:
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


def run_cpu_finetune(
    train_samples: int = 240,
    test_samples: int = 120,
    ft_epochs: int = 2,
    batch_size: int = 32,
) -> None:
    start_time = time.perf_counter()

    # In Colab, if using Drive: project_dir = Path("/content/drive/MyDrive/DL_Lab/finetune_pets")
    project_dir = Path.cwd()
    data_dir = project_dir / "data" / "raw"
    out_dir = project_dir / "outputs"
    data_dir.mkdir(parents=True, exist_ok=True)
    out_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cpu")
    torch.set_num_threads(max(1, min(4, torch.get_num_threads())))
    print(f"Device: {device} | CPU threads: {torch.get_num_threads()}")

    weights = models.ResNet50_Weights.DEFAULT
    eval_transform = weights.transforms()

    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    full_train_ds, full_test_ds = load_oxford_pet_datasets(data_dir, train_transform, eval_transform)

    train_ds = make_subset(full_train_ds, max_samples=train_samples, seed=42)
    test_ds = make_subset(full_test_ds, max_samples=test_samples, seed=42)

    num_classes = len(full_train_ds.classes)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    print(f"Classes: {num_classes} | Train subset: {len(train_ds)} | Test subset: {len(test_ds)}")

    model = models.resnet50(weights=weights)
    for param in model.parameters():
        param.requires_grad = False

    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    # Phase 1: Train classifier head (1 epoch)
    optimizer_head = torch.optim.AdamW(model.fc.parameters(), lr=1e-3)
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_head.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer_head.step()

    head_loss, head_acc = evaluate(model, test_loader, criterion, device)
    print(f"After Phase 1 (Head) | Test Loss: {head_loss:.4f} | Test Acc: {head_acc:.4f}")

    # Phase 2: Fine-tune layer4 + fc
    for param in model.layer4.parameters():
        param.requires_grad = True

    optimizer_ft = torch.optim.AdamW([
        {"params": model.layer4.parameters(), "lr": 1e-4},
        {"params": model.fc.parameters(), "lr": 5e-4},
    ])

    best_test_acc = head_acc
    ckpt_path = out_dir / "resnet50_pets_cpu_fast.pt"
    torch.save(model.state_dict(), ckpt_path)

    for epoch in range(1, ft_epochs + 1):
        model.train()
        tr_correct, tr_total = 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer_ft.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer_ft.step()
            tr_correct += (outputs.argmax(dim=1) == labels).sum().item()
            tr_total += labels.size(0)

        train_acc = tr_correct / max(tr_total, 1)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        if test_acc >= best_test_acc:
            best_test_acc = test_acc
            torch.save(model.state_dict(), ckpt_path)

        elapsed = time.perf_counter() - start_time
        print(
            f"Epoch {epoch}/{ft_epochs} ({elapsed:.1f}s) | "
            f"Train Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}"
        )

    total_time = time.perf_counter() - start_time
    print(f"Done in {total_time:.1f}s | Best Test Acc: {best_test_acc:.4f} | Saved: {ckpt_path}")


if __name__ == "__main__":
    run_cpu_finetune()

Device: cpu | CPU threads: 1
Oxford-IIIT Pet already present at /content/data/raw: True
Classes: 37 | Train subset: 240 | Test subset: 120
After Phase 1 (Head) | Test Loss: 3.3997 | Test Acc: 0.3250
Epoch 1/2 (222.2s) | Train Acc: 0.2917 | Test Loss: 3.1666 | Test Acc: 0.3917
Epoch 2/2 (332.8s) | Train Acc: 0.5833 | Test Loss: 2.7015 | Test Acc: 0.4833
Done in 332.8s | Best Test Acc: 0.4833 | Saved: /content/outputs/resnet50_pets_cpu_fast.pt
